In [6]:
import numpy as np
import scipy.sparse as sp
import random
import sys

# Function to load user-movie rating data
def load_data(file_path):
    data = np.load(file_path)  # Load the .npy file
    print("Data loaded:", data)  # Debug: Print data to confirm it's loaded
    return data

# Create sparse user-movie matrix
def create_sparse_matrix(data):
    num_users = np.max(data[:, 0])  # maximum user id
    num_movies = np.max(data[:, 1])  # maximum movie id
    print(f"Number of users: {num_users}, Number of movies: {num_movies}")  # Debug print
    # Create the sparse matrix (rows are users, columns are movies)
    user_movie_matrix = sp.csr_matrix((np.ones(len(data)), (data[:, 0] - 1, data[:, 1] - 1)),
                                      shape=(num_users, num_movies))
    print(f"Sparse matrix shape: {user_movie_matrix.shape}")  # Debug print
    return user_movie_matrix

# Minhashing: Generate signature matrix using random permutations
def minhash(user_movie_matrix, num_hashes):
    num_users, num_movies = user_movie_matrix.shape
    signature_matrix = np.full((num_hashes, num_users), np.inf)
    
    # Generate random permutations of movies for each hash function
    random_permutations = np.random.randint(0, num_movies, size=(num_hashes, num_movies))
    
    print("Random permutations for minhash:", random_permutations)  # Debug print
    
    for user_id in range(num_users):
        user_ratings = user_movie_matrix.getrow(user_id).toarray().flatten()
        
        # Apply minhash function for each permutation
        for i in range(num_hashes):
            for movie_id in random_permutations[i]:
                if user_ratings[movie_id] == 1:  # User has rated this movie
                    signature_matrix[i, user_id] = min(signature_matrix[i, user_id], movie_id)
                    break  # First rated movie in the permuted order
    print("Signature matrix after minhashing:", signature_matrix)  # Debug print
    return signature_matrix

# Banding: Split signature matrix into bands and find candidate pairs
def banding(signature_matrix, b, r):
    num_users = signature_matrix.shape[1]
    candidate_pairs = set()

    # Loop through bands
    for band in range(b):
        start_row = band * r
        end_row = (band + 1) * r
        band_signatures = signature_matrix[start_row:end_row, :]
        
        # Group users with the same signature in the band
        signature_to_users = {}
        for user_id in range(num_users):
            band_tuple = tuple(band_signatures[:, user_id])
            if band_tuple not in signature_to_users:
                signature_to_users[band_tuple] = []
            signature_to_users[band_tuple].append(user_id)

        # Add pairs of users in the same band to candidate pairs
        for users in signature_to_users.values():
            if len(users) > 1:
                for i in range(len(users)):
                    for j in range(i + 1, len(users)):
                        candidate_pairs.add((users[i], users[j]))

    print(f"Number of candidate pairs after banding: {len(candidate_pairs)}")  # Debug print
    return candidate_pairs

# Jaccard Similarity Calculation
def jaccard_similarity(user1, user2, user_movie_matrix):
    user1_ratings = user_movie_matrix.getrow(user1).toarray().flatten()
    user2_ratings = user_movie_matrix.getrow(user2).toarray().flatten()
    
    intersection = np.sum(np.logical_and(user1_ratings == 1, user2_ratings == 1))
    union = np.sum(np.logical_or(user1_ratings == 1, user2_ratings == 1))
    
    print(f"Jaccard similarity between user {user1} and user {user2}: {intersection} / {union}")  # Debug print
    
    return intersection / union if union > 0 else 0.0

# Write results to file
def write_results(candidate_pairs, user_movie_matrix, threshold=0.5):
    with open("result.txt", "w") as f:
        for u1, u2 in candidate_pairs:
            similarity = jaccard_similarity(u1, u2, user_movie_matrix)
            if similarity > threshold:
                f.write(f"{u1 + 1},{u2 + 1}\n")  # Convert to 1-based indexing
                print(f"Writing pair ({u1 + 1}, {u2 + 1}) with similarity: {similarity}")  # Debug print

# Main function to implement the LSH algorithm
def main():
    print("Starting the LSH algorithm...")  # Debug print
    data = load_data('user_movie_rating.npy')  # Load data
    user_movie_matrix = create_sparse_matrix(data)  # Create sparse matrix
    num_hashes = 100
    b = 3
    r = 10
    threshold = 0.3
    
    signature_matrix = minhash(user_movie_matrix, num_hashes)  # Generate signature matrix
    candidate_pairs = banding(signature_matrix, b, r)  # Apply banding
    write_results(candidate_pairs, user_movie_matrix, threshold)  # Write results to file

if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1].isdigit():
        seed = int(sys.argv[1])  # Accept seed as command-line argument
    else:
        seed = 42  # Default seed if no argument is provided or invalid argument
    np.random.seed(seed)  # Set the random seed for reproducibility
    main()


Starting the LSH algorithm...
Data loaded: [[     1     30      3]
 [     1    157      3]
 [     1    173      4]
 ...
 [103703  17622      2]
 [103703  17627      4]
 [103703  17764      4]]
Number of users: 103703, Number of movies: 17770
Sparse matrix shape: (103703, 17770)
Random permutations for minhash: [[15795   860  5390 ...  5761   688  6306]
 [ 5072 17739 10909 ... 14558 15383 16671]
 [  516  2201  6577 ...   206  4884  3236]
 ...
 [13893  6573  7056 ... 14226 16622 13764]
 [ 2027  2689  6715 ... 15232 17132 16785]
 [14379  2500  3115 ...  1299  3880  6499]]
Signature matrix after minhashing: [[15787.  5390.  6265. ... 15787.  8392. 15787.]
 [11924. 15955. 15955. ... 16383. 12731. 12731.]
 [ 1560.  8104.  1560. ... 13725. 13762.  4726.]
 ...
 [ 7056.  7056.  7056. ... 17153. 17153.  7056.]
 [ 4521.  6715.  2689. ...  2689. 13609. 11780.]
 [ 3647. 14962.  3147. ...  3647.  4018. 14312.]]
Number of candidate pairs after banding: 3929
Jaccard similarity between user 15306 and u

In [3]:
!pip install datasketch

In [ ]:
import numpy as np
import random
import hashlib
import scipy.sparse as sp
from datasketch import MinHash, MinHashLSH

# Function to load user-movie rating data from a .npy file
def load_data(file_path):
    try:
        data = np.load(file_path)  # Load the .npy file
        print(f"Data loaded successfully with shape {data.shape}")
        return data
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

# Create sparse user-movie matrix (using COO format)
def create_sparse_matrix(data):
    num_users = np.max(data[:, 0])  # Maximum user id
    num_movies = np.max(data[:, 1])  # Maximum movie id
    print(f"Number of users: {num_users}, Number of movies: {num_movies}")  
    # Create the sparse matrix in COO format
    user_movie_matrix = sp.coo_matrix((np.ones(len(data)), (data[:, 0] - 1, data[:, 1] - 1)),
                                      shape=(num_users, num_movies))
    print(f"Sparse matrix shape: {user_movie_matrix.shape}")  
    return user_movie_matrix

# Murmur hash function
def murmur_hash(seed, value):
    return int(hashlib.md5(f'{seed}{value}'.encode('utf-8')).hexdigest(), 16)

# Minhashing: Generate signature matrix using hash functions (Murmur Hash)
def minhash(user_movie_matrix, num_hashes):
    num_users, num_movies = user_movie_matrix.shape
    signature_matrix = np.full((num_hashes, num_users), np.inf)

    for user_id in range(num_users):
        user_ratings = user_movie_matrix.getrow(user_id).toarray().flatten()
        for i in range(num_hashes):
            min_hash = np.inf
            for movie_id in range(num_movies):
                if user_ratings[movie_id] == 1:  # User has rated this movie
                    hash_value = murmur_hash(i, movie_id)
                    min_hash = min(min_hash, hash_value)
            signature_matrix[i, user_id] = min_hash
    return signature_matrix

# Locality-Sensitive Hashing (LSH) using datasketch for banding
def lsh_banding(user_movie_matrix, threshold=0.5):
    lsh = MinHashLSH(threshold=threshold)
    minhashes = {}

    # Generate MinHash for each user and insert into LSH
    for user_id in range(user_movie_matrix.shape[0]):
        m = MinHash()
        for movie_id in range(user_movie_matrix.shape[1]):
            if user_movie_matrix[user_id, movie_id] == 1:
                m.update(str(movie_id).encode('utf8'))
        lsh.insert(str(user_id), m)
        minhashes[user_id] = m

    # Find candidate pairs using LSH
    candidate_pairs = set()
    for user_id1 in range(user_movie_matrix.shape[0]):
        for user_id2 in lsh.query(minhashes[user_id1]):
            candidate_pairs.add((user_id1, user_id2))

    return candidate_pairs

# Vectorized Jaccard similarity calculation using sparse matrices
def jaccard_similarity(user1, user2, user_movie_matrix):
    user1_ratings = user_movie_matrix[user1].toarray().flatten()
    user2_ratings = user_movie_matrix[user2].toarray().flatten()
    intersection = np.dot(user1_ratings, user2_ratings)
    union = np.sum(np.logical_or(user1_ratings, user2_ratings))
    return intersection / union if union > 0 else 0.0

# Write results to CSV using Pandas
def write_results(candidate_pairs, user_movie_matrix, threshold=0.5):
    results = []
    for u1, u2 in candidate_pairs:
        similarity = jaccard_similarity(u1, u2, user_movie_matrix)
        if similarity > threshold:
            results.append((u1 + 1, u2 + 1, similarity))  # Convert to 1-based indexing

    # Store results in a DataFrame and write to CSV
    import pandas as pd
    df = pd.DataFrame(results, columns=['User1', 'User2', 'Similarity'])
    df.to_csv('results1.csv', index=False)
    print(f"Results saved to result.csv")

# Main function to implement the LSH algorithm
def main():
    print("Starting the LSH algorithm (alternative version)...")  
    data = load_data('user_movie_rating.npy')  # Load data from .npy file
    if data is None:
        print("Error loading data, exiting.")
        return

    user_movie_matrix = create_sparse_matrix(data)  # Create sparse matrix
    num_hashes = 100
    threshold = 0.3

    signature_matrix = minhash(user_movie_matrix, num_hashes)  # Generate signature matrix
    candidate_pairs = lsh_banding(user_movie_matrix, threshold)  # Apply LSH banding
    write_results(candidate_pairs, user_movie_matrix, threshold)  # Write results to CSV

if __name__ == "__main__":
    main()


Starting the LSH algorithm (alternative version)...
Data loaded successfully with shape (65225506, 3)
Number of users: 103703, Number of movies: 17770
Sparse matrix shape: (103703, 17770)
